In [1]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool


CONFIG = {
    "root": Path("FGVD_Graph_Handover"),
    "deep_subdir": "multilevel",
    "level": "L1",  # choose from: L1, L2, L3
    "case": "all",  # choose from: all, tw_vs_all, thw_vs_all, tw_thw_vs_all
    "epochs": 25,
    "batch_size": 16,
    "lr": 1e-3,
    "weight_decay": 5e-4,
    "hidden_dim": 64,
    "heads": 8,
    "dropout": 0.5,
    "label_smoothing": 0.05,
    "patience": 5,
    "lr_factor": 0.5,
    "lr_patience": 2,
    "num_workers": 4,
    "seed": 42,
    "device": "cuda",  # "cuda" or "cpu"
    "checkpoint_dir": Path("checkpoints") / "gat",
}


def seed_everything(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG["seed"])
print("Config loaded. Update CONFIG values before training if needed.")

Config loaded. Update CONFIG values before training if needed.


In [2]:
def compute_class_weights(labels: list[int], num_classes: int) -> torch.Tensor:
    counts = np.bincount(np.asarray(labels, dtype=np.int64), minlength=num_classes).astype(np.float32)
    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def case_map_l1(l1_value: str, case_name: str) -> str:
    l1 = str(l1_value).strip().lower()
    tw_set = {"motorcycle", "scooter"}
    thw_set = {"autorickshaw", "auto", "three_wheeler", "threewheeler"}

    if case_name == "all":
        return l1_value
    if case_name == "tw_vs_all":
        return "two_wheeler" if l1 in tw_set else "other"
    if case_name == "thw_vs_all":
        return "three_wheeler" if l1 in thw_set else "other"
    if case_name == "tw_thw_vs_all":
        return "two_or_three_wheeler" if (l1 in tw_set or l1 in thw_set) else "other"
    raise ValueError(f"Unknown case: {case_name}")


@dataclass
class SampleRow:
    vehicle_id: str
    split: str
    y: int
    y_raw: str


class FGVDGraphDataset(torch.utils.data.Dataset):
    def __init__(
        self,
        rows: list[SampleRow],
        deep_dir: Path,
        adj_dir: Path,
        transform: Callable[[Data], Data] | None = None,
    ) -> None:
        self.rows = rows
        self.deep_dir = deep_dir
        self.adj_dir = adj_dir
        self.transform = transform

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, idx: int) -> Data:
        row = self.rows[idx]

        x_np = np.load(self.deep_dir / row.split / f"{row.vehicle_id}.npy").astype(np.float32)
        adj = sp.load_npz(self.adj_dir / row.split / f"{row.vehicle_id}.npz").tocoo()

        # Use torch.tensor() for robust conversion across NumPy/PyTorch builds.
        x = torch.tensor(np.asarray(x_np), dtype=torch.float32)
        edge_index = torch.tensor(np.vstack([adj.row, adj.col]), dtype=torch.long)
        edge_attr = torch.tensor(np.asarray(adj.data), dtype=torch.float32).view(-1, 1)

        data = Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            y=torch.tensor(row.y, dtype=torch.long),
            vehicle_id=row.vehicle_id,
            split=row.split,
        )

        if self.transform is not None:
            data = self.transform(data)
        return data


class GATClassifier(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        num_classes: int,
        heads: int,
        dropout: float,
    ) -> None:
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(
            in_channels=in_dim,
            out_channels=hidden_dim,
            heads=heads,
            dropout=dropout,
            edge_dim=1,
            concat=True,
        )
        self.conv2 = GATConv(
            in_channels=hidden_dim * heads,
            out_channels=hidden_dim,
            heads=1,
            dropout=dropout,
            edge_dim=1,
            concat=True,
        )
        self.norm1 = nn.BatchNorm1d(hidden_dim * heads)
        self.norm2 = nn.BatchNorm1d(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index, edge_attr, batch = data.x, data.edge_index, data.edge_attr, data.batch

        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index, edge_attr=edge_attr)
        x = self.norm1(x)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index, edge_attr=edge_attr)
        x = self.norm2(x)
        x = F.elu(x)

        g = global_mean_pool(x, batch)
        logits = self.classifier(g)
        return logits


print("Dataset and model classes ready.")

Dataset and model classes ready.


In [3]:
def build_label_enc_and_rows(
    metadata: pd.DataFrame,
    level: str,
    case_name: str,
    deep_dir: Path,
    adj_dir: Path,
) -> tuple[LabelEncoder, list[SampleRow], list[SampleRow], list[SampleRow]]:
    if level not in {"L1", "L2", "L3"}:
        raise ValueError(f"Unsupported level: {level}")

    def compute_target(df: pd.DataFrame) -> np.ndarray:
        if case_name == "all":
            return df[level].astype(str).values
        return np.asarray([case_map_l1(v, case_name) for v in df["L1"].astype(str).values])

    train_df = metadata[metadata["split"] == "train"].copy()
    val_df = metadata[metadata["split"] == "val"].copy()
    test_df = metadata[metadata["split"] == "test"].copy()

    y_train_raw = compute_target(train_df)
    y_val_raw = compute_target(val_df)
    y_test_raw = compute_target(test_df)

    encoder = LabelEncoder()
    encoder.fit(y_train_raw)
    known = set(encoder.classes_.tolist())

    def to_rows(df: pd.DataFrame, y_raw: np.ndarray) -> list[SampleRow]:
        label_mask = np.asarray([lbl in known for lbl in y_raw], dtype=bool)
        dropped_unseen = int((~label_mask).sum())

        if dropped_unseen > 0:
            split_name = str(df.iloc[0]["split"]) if len(df) else "unknown"
            print(f"Dropped {dropped_unseen} rows with unseen labels for split={split_name}")

        df = df.loc[label_mask]
        y_raw = y_raw[label_mask]
        y = encoder.transform(y_raw)

        rows: list[SampleRow] = []
        missing_deep = 0
        missing_adj = 0

        for r, label, raw in zip(df.itertuples(index=False), y, y_raw):
            deep_path = deep_dir / r.split / f"{r.vehicle_id}.npy"
            adj_path = adj_dir / r.split / f"{r.vehicle_id}.npz"
            if not deep_path.exists():
                missing_deep += 1
                continue
            if not adj_path.exists():
                missing_adj += 1
                continue
            rows.append(SampleRow(vehicle_id=r.vehicle_id, split=r.split, y=int(label), y_raw=str(raw)))

        if missing_deep > 0 or missing_adj > 0:
            split_name = str(df.iloc[0]["split"]) if len(df) else "unknown"
            print(
                f"Dropped missing files for split={split_name}: "
                f"deep={missing_deep}, adj={missing_adj}"
            )

        return rows

    return (
        encoder,
        to_rows(train_df, y_train_raw),
        to_rows(val_df, y_val_raw),
        to_rows(test_df, y_test_raw),
    )


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * batch.num_graphs
        pred = logits.argmax(dim=1)
        total_correct += int((pred == batch.y).sum().item())
        total_graphs += int(batch.num_graphs)

    return total_loss / max(total_graphs, 1), total_correct / max(total_graphs, 1)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
    device: torch.device,
) -> tuple[float, float, float, np.ndarray, np.ndarray]:
    model.eval()
    total_loss = 0.0
    all_pred: list[np.ndarray] = []
    all_true: list[np.ndarray] = []
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        loss = criterion(logits, batch.y)

        total_loss += float(loss.item()) * batch.num_graphs
        total_graphs += int(batch.num_graphs)

        pred = logits.argmax(dim=1).cpu().numpy()
        true = batch.y.cpu().numpy()
        all_pred.append(pred)
        all_true.append(true)

    y_pred = np.concatenate(all_pred) if all_pred else np.array([], dtype=np.int64)
    y_true = np.concatenate(all_true) if all_true else np.array([], dtype=np.int64)

    acc = accuracy_score(y_true, y_pred) if y_true.size > 0 else 0.0
    macro_f1 = f1_score(y_true, y_pred, average="macro") if y_true.size > 0 else 0.0
    avg_loss = total_loss / max(total_graphs, 1)
    return avg_loss, float(acc), float(macro_f1), y_true, y_pred


print("Training and evaluation utilities ready.")

Training and evaluation utilities ready.


In [4]:
def run_experiment(config: dict) -> dict:
    root = Path(config["root"])
    metadata_path = root / "metadata.csv"
    deep_dir = root / "deep_features" / config["deep_subdir"]
    adj_dir = root / "per_sample_adj"

    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing metadata file: {metadata_path}")
    if not deep_dir.exists():
        raise FileNotFoundError(f"Missing deep feature directory: {deep_dir}")
    if not adj_dir.exists():
        raise FileNotFoundError(f"Missing per-sample adjacency directory: {adj_dir}")

    metadata = pd.read_csv(metadata_path)
    encoder, train_rows, val_rows, test_rows = build_label_enc_and_rows(
        metadata=metadata,
        level=config["level"],
        case_name=config["case"],
        deep_dir=deep_dir,
        adj_dir=adj_dir,
    )

    print(f"Rows: train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")
    print(f"Classes ({len(encoder.classes_)}): {list(encoder.classes_)}")

    if len(train_rows) == 0 or len(val_rows) == 0 or len(test_rows) == 0:
        raise RuntimeError("One of the dataset splits is empty after filtering.")

    train_ds = FGVDGraphDataset(train_rows, deep_dir=deep_dir, adj_dir=adj_dir)
    val_ds = FGVDGraphDataset(val_rows, deep_dir=deep_dir, adj_dir=adj_dir)
    test_ds = FGVDGraphDataset(test_rows, deep_dir=deep_dir, adj_dir=adj_dir)

    train_loader = DataLoader(
        train_ds,
        batch_size=int(config["batch_size"]),
        shuffle=True,
        num_workers=int(config["num_workers"]),
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=int(config["num_workers"]),
        pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=int(config["batch_size"]),
        shuffle=False,
        num_workers=int(config["num_workers"]),
        pin_memory=True,
    )

    first_data = train_ds[0]
    in_dim = int(first_data.x.shape[1])
    num_classes = len(encoder.classes_)

    if str(config["device"]).lower() == "cuda" and torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")

    print(f"Using device: {device}")
    model = GATClassifier(
        in_dim=in_dim,
        hidden_dim=int(config["hidden_dim"]),
        num_classes=num_classes,
        heads=int(config["heads"]),
        dropout=float(config["dropout"]),
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(config["lr"]),
        weight_decay=float(config["weight_decay"]),
    )

    class_weights = compute_class_weights([row.y for row in train_rows], num_classes)
    class_weights = class_weights.to(device)
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=float(config.get("label_smoothing", 0.0)),
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=float(config.get("lr_factor", 0.5)),
        patience=int(config.get("lr_patience", 2)),
        threshold=1e-4,
        min_lr=1e-6,
    )

    run_name = f"{config['level']}_{config['case']}"
    out_dir = Path(config["checkpoint_dir"]) / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    best_model_path = out_dir / "best_model.pt"
    last_model_path = out_dir / "last_model.pt"
    metrics_path = out_dir / "metrics.json"
    print(f"Class weights: {class_weights.detach().cpu().numpy().round(3).tolist()}")

    best_val_acc = -1.0
    best_val_f1 = -1.0
    history: list[dict[str, float]] = []
    stagnant_epochs = 0

    epochs = int(config["epochs"])
    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step(val_f1)

        epoch_log = {
            "epoch": float(epoch),
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "val_loss": float(val_loss),
            "val_acc": float(val_acc),
            "val_f1": float(val_f1),
        }
        history.append(epoch_log)

        print(
            f"Epoch {epoch:03d}/{epochs:03d} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}"
        )

        improved = val_f1 > best_val_f1
        if improved:
            best_val_f1 = val_f1
            best_val_acc = val_acc
            stagnant_epochs = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": epoch,
                    "val_acc": val_acc,
                    "val_f1": val_f1,
                    "classes": encoder.classes_.tolist(),
                    "level": config["level"],
                    "case": config["case"],
                },
                best_model_path,
            )
        else:
            stagnant_epochs += 1

        if stagnant_epochs >= int(config.get("patience", 5)):
            print(f"Early stopping at epoch {epoch} after {stagnant_epochs} stagnant epochs.")
            break

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "epoch": epochs,
            "classes": encoder.classes_.tolist(),
            "level": config["level"],
            "case": config["case"],
        },
        last_model_path,
    )

    best_ckpt = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_ckpt["model_state_dict"])

    test_loss, test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader, criterion, device)
    y_true_labels = encoder.inverse_transform(y_true) if y_true.size > 0 else np.array([])
    y_pred_labels = encoder.inverse_transform(y_pred) if y_pred.size > 0 else np.array([])

    report = classification_report(
        y_true_labels,
        y_pred_labels,
        digits=4,
        zero_division=0,
    )
    print("\n[Test]", f"loss={test_loss:.4f}", f"acc={test_acc:.4f}", f"macro_f1={test_f1:.4f}")
    print(report)

    payload = {
        "config": {
            "root": str(root),
            "deep_subdir": config["deep_subdir"],
            "level": config["level"],
            "case": config["case"],
            "epochs": epochs,
            "batch_size": int(config["batch_size"]),
            "lr": float(config["lr"]),
            "weight_decay": float(config["weight_decay"]),
            "hidden_dim": int(config["hidden_dim"]),
            "heads": int(config["heads"]),
            "dropout": float(config["dropout"]),
            "seed": int(config["seed"]),
            "device": str(device),
        },
        "dataset": {
            "train_samples": len(train_rows),
            "val_samples": len(val_rows),
            "test_samples": len(test_rows),
            "num_classes": len(encoder.classes_),
            "classes": encoder.classes_.tolist(),
        },
        "best_val_acc": float(best_val_acc),
        "best_val_f1": float(best_val_f1),
        "test": {
            "loss": float(test_loss),
            "acc": float(test_acc),
            "macro_f1": float(test_f1),
        },
        "history": history,
        "classification_report": report,
    }

    with metrics_path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)

    print(f"Saved best model: {best_model_path}")
    print(f"Saved last model: {last_model_path}")
    print(f"Saved metrics:    {metrics_path}")
    return payload

In [28]:
# Training cell: edit settings below, then run this cell.

# Quick smoke test (recommended first):
# CONFIG["epochs"] = 1
# CONFIG["batch_size"] = 4

# Full training example:
# CONFIG["epochs"] = 25
# CONFIG["batch_size"] = 16
# CONFIG["lr"] = 3e-4
# CONFIG["weight_decay"] = 1e-3
# CONFIG["patience"] = 5
# CONFIG["lr_patience"] = 2

CONFIG["level"] = "L1"           # L1, L2, L3
CONFIG["case"] = "all"           # all, tw_vs_all, thw_vs_all, tw_thw_vs_all
CONFIG["device"] = "cuda"        # switch to "cpu" if CUDA is unavailable

result = run_experiment(CONFIG)
print("\nFinal test metrics:", result["test"])

Dropped missing files for split=val: deep=7, adj=0
Dropped missing files for split=test: deep=1, adj=0
Rows: train=15702 val=3850 test=4890
Classes (7): ['autorickshaw', 'bus', 'car', 'mini-bus', 'motorcycle', 'scooter', 'truck']
Using device: cuda
Class weights: [0.3409999907016754, 1.031999945640564, 0.16099999845027924, 4.111999988555908, 0.23999999463558197, 0.30000001192092896, 0.8140000104904175]
Epoch 001/025 | train_loss=1.2199 train_acc=0.7709 | val_loss=1.1581 val_acc=0.7694 val_f1=0.6731
Epoch 002/025 | train_loss=1.0991 train_acc=0.8002 | val_loss=1.1575 val_acc=0.8281 val_f1=0.7376
Epoch 003/025 | train_loss=1.0809 train_acc=0.8031 | val_loss=1.1610 val_acc=0.8151 val_f1=0.7253
Epoch 004/025 | train_loss=1.0834 train_acc=0.8031 | val_loss=1.1206 val_acc=0.8192 val_f1=0.7218
Epoch 005/025 | train_loss=1.0777 train_acc=0.8067 | val_loss=1.1745 val_acc=0.8003 val_f1=0.7119
Epoch 006/025 | train_loss=1.0437 train_acc=0.8177 | val_loss=1.1110 val_acc=0.8143 val_f1=0.7272
Epoch 

In [5]:
# New cell: multi-level deep-feature training (L1 + L2 + L3) with diagnostics.
# This does NOT modify your old training cell.

from pathlib import Path


def deep_feature_sanity_check(root: Path, deep_subdir: str = "multilevel", max_files_per_split: int = 32) -> None:
    deep_root = root / "deep_features" / deep_subdir
    print(f"Deep feature root: {deep_root}")

    if not deep_root.exists():
        print("[ERROR] Deep feature directory does not exist.")
        return

    for split in ["train", "val", "test"]:
        split_dir = deep_root / split
        files = sorted(split_dir.glob("*.npy")) if split_dir.exists() else []
        if not files:
            print(f"[{split}] no feature files found")
            continue

        sample_files = files[:max_files_per_split]
        means, stds = [], []
        dims = None

        for fp in sample_files:
            arr = np.load(fp)
            if dims is None:
                dims = tuple(arr.shape)
            means.append(float(arr.mean()))
            stds.append(float(arr.std()))

        mean_of_means = float(np.mean(means))
        mean_of_stds = float(np.mean(stds))
        print(
            f"[{split}] files={len(files)} sampled={len(sample_files)} "
            f"shape={dims} mean~{mean_of_means:.5f} std~{mean_of_stds:.5f}"
        )

        if mean_of_stds < 1e-3:
            print(f"[WARN] {split} deep features have very low variance -> weak signal for GAT.")


# 1) Quick diagnostics first (helps explain poor gains vs paper baseline)
deep_feature_sanity_check(Path(CONFIG["root"]), deep_subdir="multilevel")

# 2) Multi-level run (all classes for each level)
base_cfg = dict(CONFIG)
base_cfg.update(
    {
        "deep_subdir": "multilevel",
        "epochs": 40,
        "batch_size": 16,
        "lr": 5e-4,
        "weight_decay": 1e-3,
        "dropout": 0.4,
        "heads": 8,
        "patience": 8,
        "lr_patience": 3,
        "checkpoint_dir": Path("checkpoints") / "gat_deep_multilevel",
    }
)

levels = ["L1", "L2", "L3"]
cases = {"L1": "all", "L2": "all", "L3": "all"}

all_results = []

for lvl in levels:
    cfg = dict(base_cfg)
    cfg["level"] = lvl
    cfg["case"] = cases[lvl]

    print("\n" + "=" * 80)
    print(f"Running deep-feature GAT for level={lvl}, case={cfg['case']}")
    print("=" * 80)

    out = run_experiment(cfg)
    all_results.append(
        {
            "level": lvl,
            "case": cfg["case"],
            "best_val_acc": out["best_val_acc"],
            "best_val_f1": out["best_val_f1"],
            "test_acc": out["test"]["acc"],
            "test_macro_f1": out["test"]["macro_f1"],
            "train_samples": out["dataset"]["train_samples"],
            "num_classes": out["dataset"]["num_classes"],
        }
    )

summary_df = pd.DataFrame(all_results)
print("\nMulti-level summary:")
display(summary_df)

print("\nIf deep-feature GAT is not beating the paper baseline, common causes are:")
print("1) Deep features are weak for this split/domain (check std/variance above).")
print("2) Strong class imbalance in L2/L3 hurts macro-F1 even when accuracy looks okay.")
print("3) Hyperparameters from L1 may not transfer to L2/L3; tune per level.")
print("4) Baseline comparison may use a different split/protocol than this notebook run.")

Deep feature root: FGVD_Graph_Handover/deep_features/multilevel
[train] files=15702 sampled=32 shape=(4096, 64) mean~0.32126 std~0.60804
[val] files=3850 sampled=32 shape=(4096, 64) mean~0.34284 std~0.65215
[test] files=4890 sampled=32 shape=(4096, 64) mean~0.36136 std~0.66437

Running deep-feature GAT for level=L1, case=all
Dropped missing files for split=val: deep=7, adj=0
Dropped missing files for split=test: deep=1, adj=0
Rows: train=15702 val=3850 test=4890
Classes (7): ['autorickshaw', 'bus', 'car', 'mini-bus', 'motorcycle', 'scooter', 'truck']
Using device: cuda
Class weights: [0.3409999907016754, 1.031999945640564, 0.16099999845027924, 4.111999988555908, 0.23999999463558197, 0.30000001192092896, 0.8140000104904175]
Epoch 001/040 | train_loss=1.2803 train_acc=0.7683 | val_loss=1.1478 val_acc=0.7792 val_f1=0.6880
Epoch 002/040 | train_loss=1.1053 train_acc=0.8037 | val_loss=1.1376 val_acc=0.8260 val_f1=0.7318
Epoch 003/040 | train_loss=1.0797 train_acc=0.8039 | val_loss=1.1294 va

KeyboardInterrupt: 

In [8]:
# New cell: L2 improvement recipe (diagnostics + weighted sampler + focal loss)
# Purpose: Run targeted analysis and a stronger training recipe for L2 only.

from collections import Counter
from pathlib import Path
import math
import torch
from torch.utils.data import WeightedRandomSampler

# Configuration for this L2 run (won't change your original CONFIG)
cfg = dict(CONFIG)
cfg["level"] = "L2"
cfg["case"] = "all"
cfg["deep_subdir"] = "multilevel"
cfg["epochs"] = 40
cfg["batch_size"] = 16
cfg["lr"] = 5e-4
cfg["weight_decay"] = 1e-3
cfg["hidden_dim"] = max(int(cfg.get("hidden_dim", 64)), 128)
cfg["dropout"] = 0.4
cfg["checkpoint_dir"] = Path("checkpoints") / "gat_deep_L2_improved"

root = Path(cfg["root"]) if not isinstance(cfg["root"], Path) else cfg["root"]
metadata = pd.read_csv(root / "metadata.csv")

# Build label encoders and rows for L2
encoder, train_rows, val_rows, test_rows = build_label_enc_and_rows(
    metadata=metadata,
    level=cfg["level"],
    case_name=cfg["case"],
    deep_dir=root / "deep_features" / cfg["deep_subdir"],
    adj_dir=root / "per_sample_adj",
)

print(f"L2 rows: train={len(train_rows)} val={len(val_rows)} test={len(test_rows)}")
print(f"L2 classes ({len(encoder.classes_)}): {list(encoder.classes_)}")

# Class distribution
train_labels = [r.y for r in train_rows]
count = Counter(train_labels)
class_counts = {int(k): int(v) for k, v in count.items()}
print("Top 10 classes by support:")
for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  class={encoder.inverse_transform([cls])[0]:20s} -> {cnt}")

# If extreme imbalance -> use WeightedRandomSampler
num_samples = len(train_rows)
weights = [1.0 / (class_counts[int(r.y)] + 1e-6) for r in train_rows]
weights = torch.tensor(weights, dtype=torch.double)
train_sampler = WeightedRandomSampler(weights, num_samples=num_samples, replacement=True)

# Quick check of deep feature variance (sample a few files)
deep_root = root / "deep_features" / cfg["deep_subdir"]
print(f"Checking deep features in: {deep_root}")

sample_N = min(200, len(train_rows))
means = []
stds = []
shapes = Counter()
for r in train_rows[:sample_N]:
    fp = deep_root / r.split / f"{r.vehicle_id}.npy"
    try:
        arr = np.load(fp)
    except Exception:
        continue
    means.append(float(arr.mean()))
    stds.append(float(arr.std()))
    shapes[tuple(arr.shape)] += 1

if means:
    print(f"sampled mean ~ {np.mean(means):.6f}, std ~ {np.mean(stds):.6f}, common shapes: {shapes.most_common(3)}")
    if np.mean(stds) < 1e-3:
        print("[WARN] deep features have very low variance; consider regenerating or applying stronger transforms.")
else:
    print("No deep features could be loaded for sampling; check deep_features directory.")

# Compute feature-wise normalization stats on a small sample to avoid OOM
# We'll compute per-column mean/std by sampling up to 200 vehicles
sample_arrays = []
max_sample = 200
for r in train_rows[:max_sample]:
    fp = deep_root / r.split / f"{r.vehicle_id}.npy"
    try:
        arr = np.load(fp).astype(np.float32)
    except Exception:
        continue
    # flatten per-vertex feature dimension (N_vertices, feat_dim) -> mean across vertices
    if arr.ndim == 2:
        per_vehicle = arr.mean(axis=0)
    else:
        per_vehicle = arr.reshape(-1, arr.shape[-1]).mean(axis=0)
    sample_arrays.append(per_vehicle)

if sample_arrays:
    stacked = np.stack(sample_arrays, axis=0)
    feat_mean = stacked.mean(axis=0)
    feat_std = stacked.std(axis=0)
    feat_std[feat_std == 0] = 1.0
    print(f"Per-feature mean/std shapes: {feat_mean.shape}, {feat_std.shape}")
else:
    feat_mean = None
    feat_std = None
    print("Could not compute normalization stats; proceeding without normalization.")

# Define a transform to normalize Data.x per-feature (using precomputed means)
def make_normalize_transform(mean_vec, std_vec):
    def transform(data: Data):
        if mean_vec is None or std_vec is None:
            return data
        x = data.x
        # if x is 2D (n_nodes, feat_dim) normalize per feature
        if x.ndim == 2 and x.shape[1] == len(mean_vec):
            mean_t = torch.as_tensor(np.asarray(mean_vec), dtype=x.dtype, device=x.device)
            std_t = torch.as_tensor(np.asarray(std_vec), dtype=x.dtype, device=x.device)
            x = (x - mean_t) / (std_t + 1e-6)
            data.x = x
        else:
            # fallback: normalize using global mean/std scalars
            data.x = (x - x.mean()) / (x.std() + 1e-6)
        return data
    return transform

normalize_transform = make_normalize_transform(feat_mean, feat_std)

# Build datasets with transform
train_ds = FGVDGraphDataset(train_rows, deep_dir=deep_root, adj_dir=root / "per_sample_adj", transform=normalize_transform)
val_ds = FGVDGraphDataset(val_rows, deep_dir=deep_root, adj_dir=root / "per_sample_adj", transform=normalize_transform)

# Build loaders (train uses sampler)
train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), sampler=train_sampler, num_workers=int(cfg.get("num_workers", 4)))
val_loader = DataLoader(val_ds, batch_size=int(cfg["batch_size"]), shuffle=False, num_workers=int(cfg.get("num_workers", 4)))

# Focal loss implementation
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: torch.Tensor | None = None, reduction: str = "mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        if self.reduction == "mean":
            return focal.mean()
        if self.reduction == "sum":
            return focal.sum()
        return focal

# Prepare model and training state
device = torch.device("cuda" if torch.cuda.is_available() and str(cfg.get("device","")).lower()=="cuda" else "cpu")
print(f"Using device={device}")

first_data = train_ds[0]
in_dim = int(first_data.x.shape[1])
num_classes = len(encoder.classes_)
model = GATClassifier(in_dim=in_dim, hidden_dim=int(cfg["hidden_dim"]), num_classes=num_classes, heads=int(cfg.get("heads",8)), dropout=float(cfg.get("dropout",0.4))).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"]))
class_weights = compute_class_weights([r.y for r in train_rows], num_classes).to(device)
criterion = FocalLoss(gamma=2.0, weight=class_weights)

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6)

# Training loop (short, safe-run; tune epochs as needed)
best_val_f1 = -1.0
best_ckpt_dir = Path(cfg["checkpoint_dir"]) / f"{cfg['level']}_{cfg['case']}"
best_ckpt_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, int(cfg["epochs"]) + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer, lambda x,y: criterion(x,y), device)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, lambda x,y: criterion(x,y), device)
    scheduler.step(val_f1)
    print(f"Epoch {epoch}/{cfg['epochs']} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_f1={val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save({"model_state_dict": model.state_dict(), "epoch": epoch, "val_f1": val_f1}, best_ckpt_dir / "best_model.pt")

# Final evaluation on test set using best model
best_ckpt = torch.load(best_ckpt_dir / "best_model.pt", map_location=device)
model.load_state_dict(best_ckpt["model_state_dict"])

# Build test dataset and loader
test_ds = FGVDGraphDataset(test_rows, deep_dir=deep_root, adj_dir=root / "per_sample_adj", transform=normalize_transform)
test_loader = DataLoader(test_ds, batch_size=int(cfg["batch_size"]), shuffle=False, num_workers=int(cfg.get("num_workers",4)))

test_loss, test_acc, test_f1, y_true, y_pred = evaluate(model, test_loader, lambda x,y: criterion(x,y), device)
print(f"\n[Improved L2 Test] loss={test_loss:.4f} acc={test_acc:.4f} macro_f1={test_f1:.4f}")

# Save metrics summary
metrics_path = best_ckpt_dir / "metrics_improved_L2.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump({"test_loss": test_loss, "test_acc": test_acc, "test_macro_f1": test_f1}, f, indent=2)

print(f"Saved improved L2 metrics: {metrics_path}")

# Quick recommendations printed for the user
print("\nRecommendations to further improve L2 accuracy:")
print("- Increase training epochs and tune learning rate schedule per level.")
print("- Try stronger feature normalization or re-extract deep features with a different backbone.")
print("- Consider oversampling rare classes, or use class-balanced loss / focal loss (implemented above).")
print("- Perform hyperparameter search on hidden dim, heads, dropout, and weight decay.")
print("- Validate that train/val/test splits match the protocol used in the paper.")


Dropped missing files for split=val: deep=7, adj=0
Dropped missing files for split=test: deep=1, adj=0
L2 rows: train=15702 val=3850 test=4890
L2 classes (39): ['AshokLeyland', 'Atul', 'Audi', 'Bajaj', 'BharatBenz', 'Bmw', 'Chevrolet', 'Covered', 'Eicher', 'Fiat', 'Force', 'Ford', 'Hero', 'HeroHonda', 'Hindustan', 'Honda', 'Hyundai', 'Jeep', 'KTM', 'Mahindra', 'MarutiSuzuki', 'Mercedes-Benz', 'Nissan', 'Others', 'Piaggio', 'Renault', 'RoyalEnfield', 'SML', 'Skoda', 'Suzuki', 'TVS', 'Tata', 'TataMotors', 'Toyota', 'Vespa', 'Volkswagen', 'Volvo', 'Yamaha', 'bus']
Top 10 classes by support:
  class=Others               -> 2795
  class=Honda                -> 2603
  class=MarutiSuzuki         -> 1862
  class=Bajaj                -> 1445
  class=Hero                 -> 1145
  class=bus                  -> 797
  class=Toyota               -> 745
  class=TVS                  -> 650
  class=TataMotors           -> 572
  class=Hyundai              -> 540
Checking deep features in: FGVD_Graph_Ha

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.21 GiB. GPU 0 has a total capacity of 11.63 GiB of which 246.44 MiB is free. Process 1227856 has 1.11 GiB memory in use. Including non-PyTorch memory, this process has 10.26 GiB memory in use. Of the allocated memory 9.59 GiB is allocated by PyTorch, and 549.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)